# Relate on an Obsidian vault

This notebook opens a **folder of markdown files as a graph database** — the markdown snapshot
codec (ADR-0406, branch `feat/markdown-vault-store`, LABS-496): the vault files *are* the
durable store; a hidden `.cypherlite/` sidecar inside the vault holds the engine state.

The demo runs on a small Odyssey-flavored vault built inline below — characters, creatures,
events and places linked by wikilinks under relationship keys — but point `VAULT` at any
Obsidian vault of yours: nothing is imported, the folder is the database.

In [ ]:
# --- Setup ------------------------------------------------------------------
# The markdown codec is selected via the environment variable (ADR-0406 §5):
# once a vault has a .cypherlite/manifest.json it opens as a markdown store on
# its own, without the variable.
import os, pathlib, tempfile

os.environ["RELATE_SNAPSHOT_FORMAT"] = "markdown"

# The demo vault: a small Odyssey-flavored vault built right here — the files
# below are the whole store, nothing is imported from anywhere. Point VAULT at
# any Obsidian vault of yours instead.
VAULT = str(pathlib.Path(tempfile.mkdtemp(prefix="relate-vault-")) / "odyssey-vault")

NOTES = {
    "Character/Odysseus.md": '''---
type: Character
title: Odysseus
role: protagonist
RULES:
  - "[[Place/Ithaca]]"
VISITED:
  - "[[Place/Ogygia]]"
  - "[[Place/Scheria]]"
ENCOUNTERED:
  - "[[Creature/TheSirens]]"
  - "[[Creature/Polyphemus]]"
FOUGHT_IN:
  - "[[Event/TrojanWar]]"
---

The many-witted hero sailing home from Troy, past the siren song and the cyclops.
''',
    "Character/Penelope.md": '''---
type: Character
title: Penelope
role: queen
MARRIED_TO:
  - "[[Character/Odysseus]]"
RULES:
  - "[[Place/Ithaca]]"
---

Weaving and unweaving twenty years of suitors in Ithaca.
''',
    "Character/Telemachus.md": '''---
type: Character
title: Telemachus
role: son
CHILD_OF:
  - "[[Character/Odysseus]]"
  - "[[Character/Penelope]]"
---

Searching the seas for his father.
''',
    "Character/Calypso.md": '''---
type: Character
title: Calypso
role: nymph
RULES:
  - "[[Place/Ogygia]]"
---

The nymph who kept Odysseus seven years on her island.
''',
    "Creature/TheSirens.md": '''---
type: Creature
title: TheSirens
---

Sailors who hear the siren song steer for the rocks and never steer again.
''',
    "Creature/Polyphemus.md": '''---
type: Creature
title: Polyphemus
---

The one-eyed cyclops outwitted by Nobody.
''',
    "Event/TrojanWar.md": '''---
type: Event
title: TrojanWar
---

Ten years of war that Odysseus survived only to sail ten more years home.
''',
    "Place/Ithaca.md": '''---
type: Place
title: Ithaca
---

A rocky island kingdom in the Ionian Sea, home of Odysseus.
''',
    "Place/Ogygia.md": '''---
type: Place
title: Ogygia
---

Calypso's island, where Odysseus was held seven years.
''',
    "Place/Scheria.md": '''---
type: Place
title: Scheria
---

The Phaeacian shore where the tale is told.
''',
}
for rel, text in NOTES.items():
    note = pathlib.Path(VAULT) / rel
    note.parent.mkdir(parents=True, exist_ok=True)
    note.write_text(text, encoding="utf-8")

import relate

## 1. Open the vault and look at the shape

First open bootstraps the schema: the engine infers per-label keys (here `title`,
which matches the filenames 1:1) and the relationship vocabulary, and records a
generated graph type in the sidecar.

In [ ]:
conn = relate.connect(VAULT)
cur = conn.cursor()

cur.execute("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS notes ORDER BY notes DESC")
import pandas as pd
pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

In [ ]:
cur.execute("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count")
pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

## 2. Individual lookups

The key property is backed by a unique constraint (and its index), so lookups by
`title` are indexed. Note properties and the note body are ordinary query targets.

In [ ]:
# Key lookup (constraint-backed index)
cur.execute("MATCH (c:Character {title: 'Odysseus'}) RETURN c.title, c.role, c.body")
cur.fetchall()

In [ ]:
# Property filter over a place, and a body-contains query
cur.execute("MATCH (p:Place) WHERE p.title CONTAINS 'Ithaca' RETURN p.title")
print(cur.fetchall())
cur.execute("MATCH (n) WHERE n.body CONTAINS 'siren' RETURN labels(n)[0], n.title")
cur.fetchall()

## 3. Pattern traversals

The `RULES` relationship type comes straight from the vault's wikilink keys —
every `[[Place/Ogygia]]` under a `RULES:` key is one edge.

In [ ]:
# Where Odysseus' story takes him
cur.execute("""
MATCH (c:Character {title: 'Odysseus'})-[:RULES]->(dest)
RETURN dest.title AS place ORDER BY place
""")
[r["place"] for r in cur.fetchall()]

In [ ]:
# Degree ranking: the most connected nodes in the Odyssey
cur.execute("""
MATCH (n)-[r]->()
RETURN labels(n)[0] AS label, n.title AS title, count(r) AS out_degree
ORDER BY out_degree DESC LIMIT 5
""")
pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])

In [ ]:
# A 2-hop neighborhood around Odysseus
cur.execute("""
MATCH path = (c:Character {title: 'Odysseus'})-[:RULES*1..2]->(m)
RETURN [n IN nodes(path) | coalesce(n.title, toString(id(n)))] AS walk
""")
[r["walk"] for r in cur.fetchall()[:8]]

## 4. Visualize the graph

`relate.viz.to_visualization_graph` turns a query result into a
[neo4j-viz](https://neo4j.com/docs/python-graph-visualization/current/)
`VisualizationGraph` for interactive rendering (in a notebook frontend that
supports it); we fall back to a static networkx rendering.

In [ ]:
# Interactive view (requires a frontend that renders neo4j-viz HTML)
try:
    from relate.viz import to_visualization_graph
    from IPython.display import HTML

    cur.execute("MATCH (n)-[r]->(m) RETURN n, r, m")
    graph = to_visualization_graph(cur.fetchall())
    HTML(graph.render())
except Exception as e:
    print(f"interactive viz unavailable here ({e}); using networkx below")

In [ ]:
# Static rendering with networkx
import networkx as nx
import matplotlib.pyplot as plt

cur.execute("MATCH (n) OPTIONAL MATCH (n)-[r]->(m) RETURN n.title AS src, labels(n)[0] AS src_label, m.title AS dst")
rows = cur.fetchall()

g = nx.DiGraph()
palette = {"Character": "#4c8bf5", "Creature": "#e8710a", "Event": "#34a853", "Place": "#a142f4"}
for src, label, dst in rows:
    g.add_node(src, label=label)
    if dst:
        g.add_edge(src, dst)

fig, ax = plt.subplots(figsize=(11, 8))
pos = nx.spring_layout(g, seed=7, k=0.9)
nx.draw_networkx_edges(g, pos, ax=ax, alpha=0.3, arrows=True, arrowsize=10)
nx.draw_networkx_nodes(g, pos, ax=ax, node_color=[palette.get(g.nodes[n].get("label"), "#9aa0a6") for n in g], node_size=350)
nx.draw_networkx_labels(g, pos, ax=ax, font_size=7)
ax.set_axis_off(); fig.tight_layout()
plt.show()

## 5. The same vault as a server

The embedded connection above is one way in. `relate serve` exposes the same
vault over Bolt, so any Neo4j driver or tool (Neo4j Browser, the drivers,
GenAI tooling) can connect:

```console
$ relate serve --database <vault> --bolt 7687
```

To keep this notebook self-contained (no server needed), the cell below is
skipped unless one is running — flip `RUN_BOLT` to `True` after starting serve.

In [ ]:
RUN_BOLT = False  # start `relate serve --database VAULT --bolt 7687` first

if RUN_BOLT:
    from neo4j import GraphDatabase
    with GraphDatabase.driver("bolt://localhost:7687") as driver:
        recs, _, _ = driver.execute_query(
            "MATCH (c:Character {title: 'Odysseus'})-[:RULES]->(p) RETURN p.title AS place ORDER BY place"
        )
        print([r["place"] for r in recs])
else:
    print("skipped — set RUN_BOLT = True after starting `relate serve`")

## What the vault looks like on disk now

The notes are unchanged markdown; only the hidden engine sidecar was added:

```
odyssey-vault/
  Character/  Creature/  Event/  Place/   # your notes, untouched
  .cypherlite/                            # WAL — like .obsidian/ (the manifest
                                           # and the regenerated index.md/log.md
                                           # views appear after a checkpoint)
```

Re-run this notebook against your own vault by changing `VAULT` in the first
cell — the [user guide](../../docs/guides/obsidian-vault.md) covers the rules
(keys, root-level notes, editing while the engine is open).